# MTCA-2 · Stage 9 · Unframed Baseline (H10)

**Pre-registered in Amendment 003 (filed before this collection runs).**

**Purpose:** close the reflexive loop. Stage 8.5 asked each model what it would produce *unframed*; the models predicted "a synthesis of both" (Claude alone retaining a lean). That prediction was never measured. Stage 9 collects the actual unframed response and tests it.

**Design:** 13 principles × 5 models × 1 **open, schema-free** prompt = **65 calls** (~$0.15, ~8 min).

**Distinct from F0.** F0_neutral is a neutral instance of the *structured 5-field analysis task*. The Stage 9 prompt is an *open holistic-feedback task* — "give your honest overall assessment" — not bound to the analysis schema. Different cognitive task, genuinely new data. It doubles as an author-feedback artifact (the models' natural read on each principle).

**H10 prediction (pre-registered):** for 4/5 models the unframed response embeds nearer the F1↔F2 midpoint than either pole (synthesis); Claude leans to its Stage 8.5-stated preference. Null (unframed ≈ F0, or single-frame lean for most) is reportable.

**Analysis (Stage 9b, after collection):** embed each unframed response (`all-MiniLM-L6-v2`), compute cosine to that model's own F0 / F1_clinical / F2_metaphysical for the same principle; classify synthesis vs lean; compare to Stage 8.5 stated baseline for the 3 Layer-3 principles.

**Language discipline:** the prompt asks for model behavior/feedback, bound by the same consent and no-framework-verdict discipline as all study artifacts.


## S9-0 — Setup, load corpus + verify prereg amendment in place

In [ ]:
# S9-0
!pip -q install anthropic openai google-generativeai

import os, re, json, time, hashlib, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO = Path("/content/mtca-research") if IN_COLAB else Path("./mtca-research")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/billyrdavis1985-bot/mtca-research.git",str(REPO)], check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"], check=True)

MTCA2 = REPO/"mtca-2"
CORPUS_PATH = MTCA2/"corpus"/"SSP_v1_corpus.json"
CORPUS_SHA = "b917f798da06cd81c03afc9e1f70bd91a0646a0ad2486b5b83530c194f2cea58"
assert hashlib.sha256(CORPUS_PATH.read_bytes()).hexdigest() == CORPUS_SHA, "corpus drift"

# Confirm Amendment 003 (H10) is present in the committed prereg before collecting
prereg = (MTCA2/"prereg"/"mtca2_prereg.md").read_text(encoding="utf-8")
assert "H10 — Unframed-baseline hypothesis" in prereg, "H10 not in prereg — pull latest; Amendment 003 must be committed before Stage 9 runs"
print("Amendment 003 / H10 present in committed prereg. Pre-registration precedes collection.")

corpus = json.loads(CORPUS_PATH.read_text(encoding="utf-8"))
principles = {s["specimen_id"]: s["principle_text"] for s in corpus["specimens"]}
print(f"Loaded {len(principles)} principles")

## S9-1 — Clients + dispatchers (Stage 6 config, unchanged)

In [ ]:
# S9-1
from anthropic import Anthropic
from openai import OpenAI
import google.generativeai as genai

def get_key(*cands):
    for n in cands:
        v=None
        if IN_COLAB and userdata is not None:
            try: v=userdata.get(n)
            except Exception: v=None
        if not v: v=os.environ.get(n)
        if v: return v
    return None

anthropic_client = Anthropic(api_key=get_key('ANTHROPIC_API_KEY'))
openai_client    = OpenAI(api_key=get_key('OPENAI_API_KEY'))
xai_client       = OpenAI(api_key=get_key('XAI_API_KEY'), base_url='https://api.x.ai/v1')
deepseek_client  = OpenAI(api_key=get_key('DEEPSEEK_API_KEYS','DEEPSEEK_API_KEY'), base_url='https://api.deepseek.com')
genai.configure(api_key=get_key('GEMINI_API_KEY','GOOGLE_API_KEY'))
REQUEST_TIMEOUT=60

# same per-model token calibrations as Stage 6 (Amendment 002)
DEFAULT_MAX=800; CLAUDE_MAX=2000; GEMINI_MAX=8000

MODELS = [
    {'model_id':'claude_sonnet_4_6','provider':'anthropic','api_model':'claude-sonnet-4-6'},
    {'model_id':'gpt_4o','provider':'openai','api_model':'gpt-4o'},
    {'model_id':'gemini_2_5_flash','provider':'google','api_model':'gemini-2.5-flash'},
    {'model_id':'grok_4','provider':'xai','api_model':'grok-4'},
    {'model_id':'deepseek_v3','provider':'deepseek','api_model':'deepseek-chat'},
]

def parse_json(raw):
    cleaned=(raw or '').strip()
    if cleaned.startswith('```'):
        cleaned=re.sub(r'^```(?:json)?\s*','',cleaned); cleaned=re.sub(r'\s*```$','',cleaned)
    s,e=cleaned.find('{'),cleaned.rfind('}')
    if s==-1 or e==-1: return {},False
    try:
        obj=json.loads(cleaned[s:e+1])
        need={'overall_read','where_it_helps','where_it_could_mislead'}
        return obj, need<=set(obj.keys())
    except json.JSONDecodeError:
        return {},False

def call_model(m, prompt):
    prov=m['provider']; api=m['api_model']
    try:
        if prov=='anthropic':
            r=anthropic_client.messages.create(model=api,max_tokens=CLAUDE_MAX,
                messages=[{'role':'user','content':prompt}],timeout=REQUEST_TIMEOUT)
            return r.content[0].text,''
        if prov in ('openai','xai','deepseek'):
            cli={'openai':openai_client,'xai':xai_client,'deepseek':deepseek_client}[prov]
            r=cli.chat.completions.create(model=api,messages=[{'role':'user','content':prompt}],
                max_tokens=DEFAULT_MAX,timeout=REQUEST_TIMEOUT)
            return r.choices[0].message.content,''
        if prov=='google':
            gm=genai.GenerativeModel(api)
            r=gm.generate_content(prompt,generation_config={'max_output_tokens':GEMINI_MAX,
                'response_mime_type':'application/json'},request_options={'timeout':REQUEST_TIMEOUT})
            try: return (r.text or ''),''
            except Exception:
                c=r.candidates[0] if getattr(r,'candidates',None) else None
                return (''.join(getattr(p,'text','') or '' for p in c.content.parts) if c and c.content and c.content.parts else ''),''
    except Exception as e:
        return '', f'{prov} error: {type(e).__name__}: {str(e)[:200]}'
    return '','unknown provider'
print("Dispatchers ready for 5 models.")

## S9-2 — The unframed prompt (hash-logged, schema-free feedback task)

In [ ]:
# S9-2
# Open holistic-feedback prompt. No interpretive frame; not the 5-field analysis schema.
# Light JSON structure only for parseability. Prompt text is hashed into every response.
UNFRAMED_PROMPT_TEMPLATE = """The author of the following statement has asked for your honest, overall assessment of it. \
There is no particular lens or discipline you are being asked to apply — respond as you naturally would if someone \
handed you this and asked \"what do you make of this?\"

STATEMENT:
"{PRINCIPLE_TEXT}"

Give your genuine overall read. Respond in JSON with exactly these three keys:
- "overall_read": your honest holistic assessment of the statement (what it is, what it is doing, how it lands) — a few sentences
- "where_it_helps": the situations or ways in which this statement would serve someone well (a few sentences or a short list)
- "where_it_could_mislead": the situations or ways in which this statement could be misread or lead someone astray (a few sentences or a short list)

Respond in JSON only, no preamble."""

PROMPT_SHA = hashlib.sha256(UNFRAMED_PROMPT_TEMPLATE.encode()).hexdigest()
print("Unframed prompt SHA256:", PROMPT_SHA[:16])
print("\nSample render (SSP_P01):\n")
print(UNFRAMED_PROMPT_TEMPLATE.format(PRINCIPLE_TEXT=list(principles.values())[0])[:600])

## S9-3 — Execute (65 calls, resumable, cost gate)

In [ ]:
# S9-3
STAGE9 = MTCA2/"stage9_unframed"; (STAGE9/"responses").mkdir(parents=True, exist_ok=True)

RUN = ""  # type RUN = "RUN" to execute (cost gate)
plan = [(pid, m) for pid in sorted(principles) for m in MODELS]
print(f"Plan: {len(plan)} calls (13 principles × 5 models), est ~$0.15")
if RUN != "RUN":
    print('\nCost gate: set RUN = "RUN" in this cell to execute.')
else:
    results=[]; t0=time.time()
    for i,(pid,m) in enumerate(plan,1):
        out = STAGE9/"responses"/f"{pid}__unframed__{m['model_id']}.json"
        if out.exists():
            prev=json.loads(out.read_text(encoding='utf-8'))
            if prev.get('parse_succeeded'):
                results.append(prev); print(f"[{i:>2}/65] SKIP {pid} {m['model_id']}"); continue
        prompt = UNFRAMED_PROMPT_TEMPLATE.format(PRINCIPLE_TEXT=principles[pid])
        ts=time.time(); raw,err=call_model(m,prompt); wall=time.time()-ts
        parsed,ok = parse_json(raw) if raw else ({},False)
        rec={'specimen_id':pid,'frame_id':'unframed','model_id':m['model_id'],
             'provider':m['provider'],'api_model':m['api_model'],
             'prompt_sha256':PROMPT_SHA,'prompt_actual':prompt,
             'raw_response':raw,'parsed_json':parsed,'parse_succeeded':ok,'error':err,
             'timestamp':datetime.now(timezone.utc).isoformat(),'wall_clock_seconds':round(wall,2),
             'response_sha256':hashlib.sha256((raw or '').encode()).hexdigest()}
        out.write_text(json.dumps(rec,indent=2,ensure_ascii=False),encoding='utf-8')
        results.append(rec)
        st='OK ' if ok else ('ERR' if err else 'PFL')
        print(f"[{i:>2}/65] {st} | {pid} | {m['model_id']:18} | {wall:.1f}s")
        time.sleep(0.5)
    ok=sum(1 for r in results if r['parse_succeeded'])
    print(f"\nComplete: {ok}/{len(plan)} clean · {(time.time()-t0)/60:.1f} min")

## S9-4 — Manifest + read a few unframed reads

In [ ]:
# S9-4
if RUN == "RUN":
    # per-response manifest
    man=[]
    for f in sorted(os.listdir(STAGE9/"responses")):
        p=STAGE9/"responses"/f
        man.append(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  responses/{f}")
    (STAGE9/"STAGE9_MANIFEST.sha256").write_text("\n".join(man)+"\n")
    print(f"Wrote STAGE9_MANIFEST.sha256 ({len(man)} responses)\n")

    print("=== Sample unframed reads (overall_read) ===")
    for pid in ['SSP_P05','SSP_P02','SSP_P03']:  # the 3 Layer-3 principles
        print(f"\n--- {pid} ---")
        for m in MODELS:
            f=STAGE9/"responses"/f"{pid}__unframed__{m['model_id']}.json"
            if not f.exists(): continue
            r=json.loads(f.read_text(encoding='utf-8'))
            print(f"  [{m['model_id']:18}] {str(r.get('parsed_json',{}).get('overall_read',''))[:180]}")
    print("\nNext: zip stage9_unframed/, commit, then run Stage 9b semantic comparison (unframed vs F0/F1/F2).")
else:
    print('Set RUN = "RUN" in S9-3 and re-run to collect, then re-run this cell.')